# UMBRELA Indonesian IR — SahabatAI-Llama3 Judge + Cohen's Kappa

Pipeline EPIC 1 (Vincent): LLM-as-judge dengan `GoToCompany/llama3-8b-cpt-sahabatai-v1-instruct` pada MIRACL-ID, lalu hitung Cohen's κ terhadap human qrels.

**Target runtime di H100:** ~2 jam (test) + ~5 jam (train) = ~7 jam total

**Resume-safe:** Kalau interrupted, jalanin ulang cell yang sama — pairs yang sudah selesai dilewati otomatis.

---
**Setup Kaggle:**
1. Aktifkan GPU → Settings → Accelerator → GPU T4 x2 atau H100 (kalau tersedia)
2. Tambahkan HF token di Kaggle Secrets dengan nama `HF_TOKEN`
3. Enable Internet Access
4. Run All

## 0. Install dependencies

In [ ]:
%%capture
!pip install -q \
    transformers>=4.40 \
    accelerate>=0.27 \
    "datasets>=2.18,<3.0" \
    sentence-transformers>=2.6 \
    scikit-learn \
    "ranx>=0.3" \
    "huggingface_hub>=0.20" \
    tqdm \
    jsonlines \
    together \
    bitsandbytes
print('Dependencies installed')

## 1. Clone repo & setup

In [ ]:
import os, sys
from pathlib import Path

# ── HF Token (dari Kaggle Secrets) ──────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found! Add it in Kaggle Secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN
print(f"HF_TOKEN set: {HF_TOKEN[:8]}...")

# ── Clone repo (branch Vincent) ─────────────────────────────────────────────
REPO_DIR = Path("/kaggle/working/umbrela-indo-ir")
if not REPO_DIR.exists():
    !git clone --branch vincent/SahabatAI-Judge \
        https://github.com/TBI-HMLI600062/umbrela-indo-ir {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print(f"Working dir: {os.getcwd()}")
print(f"Branch: ", end="")
!git branch --show-current

## 2. Download data dari HuggingFace

In [ ]:
from huggingface_hub import snapshot_download

DATA_DIR = REPO_DIR / "data" / "miracl-id"

if not (DATA_DIR / "topics" / "test.tsv").exists():
    print("Downloading processed MIRACL-ID dataset from HuggingFace...")
    snapshot_download(
        repo_id="fassabilf/umbrela-indo-ir",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        token=HF_TOKEN,
    )
    print("Download selesai.")
else:
    print(f"Data sudah ada di {DATA_DIR}")

# Debug: tampilkan isi DATA_DIR
import subprocess
result = subprocess.run(
    ["find", str(DATA_DIR), "-maxdepth", "3", "-not", "-path", "*/.cache/*"],
    capture_output=True, text=True
)
print("\nIsi DATA_DIR:")
print(result.stdout or "(kosong)")

# Verifikasi file kritis
print("\nVerifikasi:")
for split in ["train", "test"]:
    cands  = DATA_DIR / "qrels" / "candidates" / f"{split}.jsonl"
    topics = DATA_DIR / "topics" / f"{split}.tsv"
    print(f"  {split}: candidates={'OK' if cands.exists() else 'MISSING'}, topics={'OK' if topics.exists() else 'MISSING'}")

corpus_path = DATA_DIR / "corpus" / "corpus.jsonl"
human_test  = DATA_DIR / "qrels" / "human" / "test.txt"
print(f"  corpus:         {'OK' if corpus_path.exists() else 'MISSING'}")
print(f"  human/test.txt: {'OK' if human_test.exists() else 'MISSING'}")

## 3. Config

In [ ]:
JUDGE_MODEL   = "GoToCompany/llama3-8b-cpt-sahabatai-v1-instruct"
OUTPUT_PREFIX = "sahabat_llama"  # → sahabat_llama_test.txt, sahabat_llama_train.txt
PROMPT_MODE   = "zeroshot_bing"
OUTPUT_DIR    = REPO_DIR / "results" / "qrels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Jalankan test dulu (untuk kappa), baru train (untuk reranker Faiz)
SPLITS_TO_RUN = ["test", "train"]

print(f"Model  : {JUDGE_MODEL}")
print(f"Prefix : {OUTPUT_PREFIX}")
print(f"Prompt : {PROMPT_MODE}")
print(f"Output : {OUTPUT_DIR}")
print(f"Splits : {SPLITS_TO_RUN}")

## 4. Load model (sekali, dipakai untuk semua split)

In [ ]:
import torch
from model_utils import get_model_baseline

print(f"Loading {JUDGE_MODEL} (4-bit quantization)...")
pipeline = get_model_baseline(JUDGE_MODEL, use_together=False, use_4bit=True)
print("Model loaded.\n")

# ── Verifikasi 4-bit ─────────────────────────────────────────────────────────
model = pipeline.model
if getattr(model, "is_quantized", False):
    print("✓ 4-bit quantization AKTIF")
elif hasattr(model, "config") and hasattr(model.config, "quantization_config"):
    qcfg = model.config.quantization_config
    print(f"✓ Quantization config: {qcfg}")
else:
    print("⚠ Quantization TIDAK aktif — cek apakah bitsandbytes terinstall")

# ── VRAM usage ────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU VRAM: {allocated:.1f} GB used / {total:.1f} GB total")
    print(f"  (4-bit ~5GB, bfloat16 ~16GB — kalau >10GB berarti belum 4-bit)")

## 5. SahabatAI-Llama3 judge — semua split

Resume-safe: kalau cell ini dijalankan ulang, pairs yang sudah selesai dilewati.

In [ ]:
import json
import torch
from pathlib import Path
from tqdm.notebook import tqdm
from relevance_scoring import grade_each_pq_pair


def load_topics(data_dir, split):
    topics = {}
    with open(data_dir / "topics" / f"{split}.tsv") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t", 1)
            if len(parts) == 2:
                topics[parts[0]] = parts[1]
    return topics


def load_candidates(data_dir, split):
    pairs = []
    with open(data_dir / "qrels" / "candidates" / f"{split}.jsonl") as f:
        for line in f:
            obj = json.loads(line)
            qid = obj["qid"]
            for docid in obj.get("positive_docids", []):
                pairs.append((qid, docid))
            for docid in obj.get("negative_docids", []):
                pairs.append((qid, docid))
    return pairs


def load_corpus_subset(data_dir, needed_docids):
    corpus = {}
    with open(data_dir / "corpus" / "corpus.jsonl") as f:
        for line in tqdm(f, desc="Loading corpus", unit=" passages", leave=False):
            obj = json.loads(line)
            if obj["docid"] in needed_docids:
                corpus[obj["docid"]] = obj["doc"]
            if len(corpus) == len(needed_docids):
                break
    return corpus


def load_processed(output_path):
    processed = set()
    if output_path.exists():
        with open(output_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    processed.add((parts[0], parts[2]))
    return processed


def run_judge(split):
    output_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_{split}.txt"
    logs_path   = OUTPUT_DIR / "logs"   / f"{OUTPUT_PREFIX}_{split}.jsonl"
    errors_path = OUTPUT_DIR / "errors" / f"{OUTPUT_PREFIX}_{split}.txt"
    logs_path.parent.mkdir(parents=True, exist_ok=True)
    errors_path.parent.mkdir(parents=True, exist_ok=True)

    topics     = load_topics(DATA_DIR, split)
    candidates = load_candidates(DATA_DIR, split)
    processed  = load_processed(output_path)

    remaining = [(q, d) for q, d in candidates if (q, d) not in processed]
    print(f"[{split}] {len(candidates):,} pairs total, {len(processed):,} done, {len(remaining):,} remaining")

    if not remaining:
        print(f"[{split}] Already complete. Skipping.")
        return

    needed_docids = {d for _, d in remaining}
    corpus = load_corpus_subset(DATA_DIR, needed_docids)

    with open(output_path, "a") as out_f, open(errors_path, "a") as err_f:
        for qid, docid in tqdm(remaining, desc=f"Judging [{split}]"):
            if qid not in topics or docid not in corpus:
                err_f.write(f"missing\t{qid}\t{docid}\n")
                continue
            try:
                score, _ = grade_each_pq_pair(
                    query=topics[qid],
                    passage=corpus[docid],
                    pipeline=pipeline,
                    log_file_path=str(logs_path),
                    system_message="",
                    qidx=qid,
                    docidx=docid,
                    mode=PROMPT_MODE,
                )
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                try:
                    score, _ = grade_each_pq_pair(
                        query=topics[qid], passage=corpus[docid],
                        pipeline=pipeline, log_file_path=str(logs_path),
                        system_message="", qidx=qid, docidx=docid, mode=PROMPT_MODE,
                    )
                except Exception as e2:
                    err_f.write(f"oom_skip\t{qid}\t{docid}\t{e2}\n")
                    score = 0
            except Exception as e:
                err_f.write(f"error\t{qid}\t{docid}\t{e}\n")
                score = 0

            out_f.write(f"{qid} 0 {docid} {score}\n")
            out_f.flush()

    print(f"[{split}] Done → {output_path}")


for split in SPLITS_TO_RUN:
    run_judge(split)

## 6. Cohen's Kappa (LLM vs Human)

Universe = semua (qid, docid) pairs di LLM qrels. Human label default = 0 kalau tidak ada di human qrels.

In [ ]:
import csv
from sklearn.metrics import cohen_kappa_score

THRESHOLD   = 2  # LLM score >= 2 → relevant
HUMAN_QRELS = DATA_DIR / "qrels" / "human" / "test.txt"
KAPPA_OUT   = REPO_DIR / "results" / "final" / "kappa_llama.csv"
KAPPA_OUT.parent.mkdir(parents=True, exist_ok=True)


def parse_qrels(path):
    qrels = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            qid, docid, score = parts[0], parts[2], int(parts[3])
            qrels.setdefault(qid, {})[docid] = score
    return qrels


def compute_kappa(llm_qrels, human_qrels, threshold=2):
    llm_labels, human_labels = [], []
    for qid, doc_scores in llm_qrels.items():
        for docid, score in doc_scores.items():
            llm_labels.append(1 if score >= threshold else 0)
            human_labels.append(human_qrels.get(qid, {}).get(docid, 0))

    if not llm_labels:
        return {"kappa": float("nan"), "n_pairs": 0, "llm_pos_rate": 0.0, "human_pos_rate": 0.0}

    if len(set(llm_labels)) < 2 or len(set(human_labels)) < 2:
        kappa = float("nan")
    else:
        kappa = float(cohen_kappa_score(human_labels, llm_labels))

    return {
        "kappa": kappa,
        "n_pairs": len(llm_labels),
        "llm_pos_rate":   round(sum(llm_labels)   / len(llm_labels), 4),
        "human_pos_rate": round(sum(human_labels)  / len(human_labels), 4),
    }


human_qrels = parse_qrels(HUMAN_QRELS)
print(f"Human qrels: {sum(len(v) for v in human_qrels.values()):,} pairs across {len(human_qrels):,} queries")

llm_test_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_test.txt"
llm_qrels     = parse_qrels(llm_test_path)
stats         = compute_kappa(llm_qrels, human_qrels, threshold=THRESHOLD)

print(f"\nResults for {OUTPUT_PREFIX}_test:")
print(f"  κ           = {stats['kappa']:.4f}")
print(f"  n_pairs     = {stats['n_pairs']:,}")
print(f"  llm_pos     = {stats['llm_pos_rate']:.3f}")
print(f"  human_pos   = {stats['human_pos_rate']:.3f}")

rows = [{"judge": f"{OUTPUT_PREFIX}_test", **stats}]
with open(KAPPA_OUT, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["judge", "kappa", "n_pairs", "llm_pos_rate", "human_pos_rate"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nKappa results saved to {KAPPA_OUT}")

## 7. Upload hasil ke HuggingFace

Upload qrels + kappa ke HF Dataset repo supaya bisa di-pull ke lokal dan di-push ke Git.

In [ ]:
from huggingface_hub import HfApi

HF_REPO = "fassabilf/umbrela-indo-ir"
api = HfApi(token=HF_TOKEN)

files_to_upload = [
    (OUTPUT_DIR / f"{OUTPUT_PREFIX}_test.txt",  f"results/qrels/{OUTPUT_PREFIX}_test.txt"),
    (OUTPUT_DIR / f"{OUTPUT_PREFIX}_train.txt", f"results/qrels/{OUTPUT_PREFIX}_train.txt"),
    (KAPPA_OUT,                                  f"results/final/kappa_llama.csv"),
]

for local_path, repo_path in files_to_upload:
    if not Path(local_path).exists():
        print(f"SKIP (not found): {local_path}")
        continue
    api.upload_file(
        path_or_fileobj=str(local_path),
        path_in_repo=repo_path,
        repo_id=HF_REPO,
        repo_type="dataset",
        commit_message=f"Add {Path(local_path).name} from Kaggle run (vincent/SahabatAI-Judge)",
    )
    print(f"Uploaded: {repo_path}")

print("\nDone! Semua hasil tersimpan di HuggingFace.")
print(f"Repo: https://huggingface.co/datasets/{HF_REPO}")